In [1]:

import os
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from collections import defaultdict
from torch.utils.data import DataLoader


In [2]:
# %%
# Cell 2: Load fold data and gene list CSV

# Define the fold folder path (using fold_1 as an example)
fold_folder = "/home/chb3333/yulab/chb3333/gem-patho/data_extraction/kfolds/jinaai_kfold_20val/fold_1"
train_path = os.path.join(fold_folder, "train.parquet")

# Use only the needed columns.
cols = ["Case ID", "gene_embed_seq", "OS.time", "OS", "type", "description_embeddings"]

# Load the training DataFrame from the fold.
train_df = pd.read_parquet(train_path, engine="pyarrow")[cols]
print("Train DataFrame shape:", train_df.shape)

# Load the combined gene list CSV.
gene_list_csv = "/home/chb3333/yulab/chb3333/gem-patho/data_extraction/cancer_gene_list_selection/combined_genelist.csv"
gene_list_df = pd.read_csv(gene_list_csv)
# Assume the CSV has a column "Gene Symbol" that lists the gene names.
gene_list = gene_list_df["Gene Symbol"].tolist()
print("Loaded gene list with", len(gene_list), "genes")


Train DataFrame shape: (7562, 6)
Loaded gene list with 1398 genes


In [3]:
# %%
# Cell 3: Define the pretrained transformer model.
# This is the same as your original model implementation.
class PreprocessedTransformerSurvivalModel(nn.Module):
    def __init__(self, d_gene, d_model=256, polyphen_hidden_dim=128, nhead=4, dropout=0.1, desc_dim=None):
        super(PreprocessedTransformerSurvivalModel, self).__init__()
        self.gene_linear = nn.Linear(d_gene, d_model)
        self.polyphen_mlp = nn.Sequential(
            nn.Linear(1, polyphen_hidden_dim),
            nn.GELU(),
            nn.Linear(polyphen_hidden_dim, d_model)
        )
        self.cna_mlp = nn.Sequential(
            nn.Linear(1, polyphen_hidden_dim),
            nn.GELU(),
            nn.Linear(polyphen_hidden_dim, d_model)
        )
        self.cancer_type_mlp = nn.Sequential(
            nn.Linear(6, polyphen_hidden_dim),
            nn.GELU(),
            nn.Linear(polyphen_hidden_dim, d_model)
        )
        if desc_dim is None:
            desc_dim = d_gene
        self.description_linear = nn.Linear(desc_dim, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                    dropout=dropout, activation="gelu")
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.final_linear = nn.Linear(d_model, 1)
        
    def forward(self, emb, scores, cnas, cancer_type, description, src_key_padding_mask=None):
        gene_proj = self.gene_linear(emb)
        polyphen_proj = self.polyphen_mlp(scores.unsqueeze(-1))
        cna_proj = self.cna_mlp(cnas.unsqueeze(-1))
        cancer_type_proj = self.cancer_type_mlp(cancer_type).unsqueeze(1)
        token_emb = gene_proj + polyphen_proj + cna_proj + cancer_type_proj
        
        desc_proj = self.description_linear(description).unsqueeze(1)
        token_emb = torch.cat([desc_proj, token_emb], dim=1)
        
        if src_key_padding_mask is not None:
            new_mask = torch.cat([torch.zeros(src_key_padding_mask.size(0), 1, device=src_key_padding_mask.device,
                                               dtype=src_key_padding_mask.dtype),
                                  src_key_padding_mask], dim=1)
        else:
            new_mask = None
        
        token_emb = token_emb.transpose(0, 1)
        transformer_out = self.transformer_encoder(token_emb, src_key_padding_mask=new_mask)
        transformer_out = transformer_out.transpose(0, 1)
        pooled = transformer_out[:, 0, :]
        risk = self.final_linear(pooled)
        return risk


In [4]:
# %%
# Cell 4: Updated Dataset & Collate Function with Gene Name Lookup
#
# This dataset extracts tokens from the "gene_embed_seq" column.
# For each token, if the "gene" field is empty, we replace it with the corresponding gene name from gene_list.
class PreprocessedSequenceDatasetWithNames(torch.utils.data.Dataset):
    def __init__(self, df, token_col="gene_embed_seq", cancer_type_mapping=None):
        self.df = df.reset_index(drop=True)
        self.token_col = token_col
        self.cancer_type_mapping = cancer_type_mapping if cancer_type_mapping is not None else {}
        self.has_description = "description_embeddings" in self.df.columns

        self.genename_dim = None
        for idx in range(len(self.df)):
            tokens = self.df.iloc[idx][token_col]
            if isinstance(tokens, np.ndarray):
                tokens = tokens.tolist()
            if tokens and len(tokens) > 0:
                self.genename_dim = len(tokens[0]["embedding"])
                break
        if self.genename_dim is None:
            raise ValueError("Could not determine gene embedding dimension from data.")

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = row[self.token_col]
        if isinstance(tokens, np.ndarray):
            tokens = tokens.tolist()
        if not tokens or (hasattr(tokens, '__len__') and len(tokens) == 0):
            tokens = [{"gene": "NA", "embedding": [0.0]*self.genename_dim, "score": 0.0, "cna": 0.0}]
        gene_names = [token.get("gene", "") for token in tokens]
        # Replace empty gene names with corresponding name from gene_list (if available).
        for i, name in enumerate(gene_names):
            if not name:
                gene_names[i] = gene_list[i] if i < len(gene_list) else "Unknown"
        embeddings = [torch.tensor(token["embedding"], dtype=torch.float) for token in tokens]
        scores = [torch.tensor(token["score"], dtype=torch.float) for token in tokens]
        cnas = [torch.tensor(token.get("cna", 0.0), dtype=torch.float) for token in tokens]
        
        cancer_type_acronym = row.get("type", None)
        if cancer_type_acronym is None or cancer_type_acronym not in self.cancer_type_mapping:
            ct_vector = [0]*6
        else:
            ct_vector = self.cancer_type_mapping[cancer_type_acronym]
        cancer_type_tensor = torch.tensor(ct_vector, dtype=torch.float)
        
        if self.has_description:
            description = torch.tensor(row["description_embeddings"], dtype=torch.float)
        else:
            description = torch.zeros(self.genename_dim, dtype=torch.float)
        
        time = torch.tensor(row["OS.time"], dtype=torch.float)
        event = torch.tensor(row["OS"], dtype=torch.float)
        case_id = row.get("Case ID", "Unknown")
        return embeddings, scores, cnas, cancer_type_tensor, description, time, event, case_id, gene_names

def collate_fn_preprocessed_with_names(batch):
    emb_list, score_list, cna_list, cancer_type_list, desc_list, times, events, case_ids, gene_names_list = zip(*batch)
    padded_emb = torch.nn.utils.rnn.pad_sequence([torch.stack(seq) for seq in emb_list],
                                                  batch_first=True, padding_value=0.0)
    padded_scores = torch.nn.utils.rnn.pad_sequence([torch.stack(seq) for seq in score_list],
                                                     batch_first=True, padding_value=0.0)
    padded_cnas = torch.nn.utils.rnn.pad_sequence([torch.stack(seq) for seq in cna_list],
                                                  batch_first=True, padding_value=0.0)
    cancer_types = torch.stack(cancer_type_list)
    descriptions = torch.stack(desc_list)
    
    lengths = torch.tensor([len(seq) for seq in emb_list], dtype=torch.long)
    B, L_max, _ = padded_emb.shape
    mask = torch.zeros((B, L_max), dtype=torch.bool)
    for i, l in enumerate(lengths):
        if l < L_max:
            mask[i, l:] = True
    times = torch.stack(times)
    events = torch.stack(events)
    return padded_emb, padded_scores, padded_cnas, cancer_types, descriptions, times, events, mask, list(case_ids), list(gene_names_list)


In [11]:
# %%
# Cell 5: Attention-Based Attributions Function
def get_attention_attributions(model, dataloader, device, cancer_type_mapping, type_to_index):
    """
    Captures attention weights from each Transformer encoder layer.
    Returns a dict mapping Case ID to a dict with:
      - 'cancer_type': integer cancer type,
      - 'gene_names': list of gene names,
      - 'attention_scores': average attention scores for each gene token.
    """
    attention_data = {}
    hook_handles = {}
    collected_attn = {}

    def get_hook(name):
        def hook(module, input, output):
            # Now output is a tuple (attn_output, attn_weights).
            # Ensure attn_weights is not None.
            if isinstance(output, tuple) and output[1] is not None:
                collected_attn[name] = output[1].detach().cpu()
            else:
                collected_attn[name] = None
        return hook

    # Register hooks for each layer in the transformer encoder.
    for i, layer in enumerate(model.transformer_encoder.layers):
        handle = layer.self_attn.register_forward_hook(get_hook(f"layer_{i}"))
        hook_handles[f"layer_{i}"] = handle

    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            emb, scores, cnas, cancer_type, description, times, events, mask, case_ids, gene_names_list = batch
            emb = emb.to(device)
            scores = scores.to(device)
            cnas = cnas.to(device)
            cancer_type = cancer_type.to(device)
            description = description.to(device)
            mask = mask.to(device)
            _ = model(emb, scores, cnas, cancer_type, description, src_key_padding_mask=mask)
            batch_size = emb.shape[0]
            for i in range(batch_size):
                sample_case = case_ids[i]
                ct_list = cancer_type[i].detach().cpu().tolist()
                sample_ct = None
                for key, binary_vec in cancer_type_mapping.items():
                    if binary_vec == [int(x) for x in ct_list]:
                        sample_ct = type_to_index[key]
                        break
                layer_attn = []
                # Only include layers that returned attention weights.
                for key in collected_attn:
                    if collected_attn[key] is not None:
                        attn_sample = collected_attn[key][i]  # shape: [num_heads, seq_len, seq_len]
                        attn_avg = attn_sample.mean(dim=0)       # average over heads
                        layer_attn.append(attn_avg)
                if layer_attn:
                    attn_avg_all = torch.stack(layer_attn).mean(dim=0)  # [seq_len, seq_len]
                    # Assume token 0 is the description; gene tokens are indices 1:.
                    if attn_avg_all.dim() >= 2:
                        gene_attn = attn_avg_all[0, 1:]
                    else:
                        gene_attn = torch.zeros(emb.shape[1]-1)
                else:
                    # If no attention weights were captured, use zeros.
                    gene_attn = torch.zeros(emb.shape[1]-1)
                attention_data[sample_case] = {
                    "cancer_type": sample_ct,
                    "gene_names": gene_names_list[i],
                    "attention_scores": gene_attn.numpy()
                }
            collected_attn.clear()

    # Remove hooks.
    for handle in hook_handles.values():
        handle.remove()

    return attention_data


In [12]:
# %%
# Cell 6: Gradient-Based Attributions Function
def get_gradient_attributions(model, dataloader, device, cancer_type_mapping, type_to_index):
    """
    Computes gradient * input for gene embeddings.
    Returns a dict mapping Case ID to a dict with:
      - 'cancer_type': integer cancer type,
      - 'gene_names': list of gene names,
      - 'gradient_attributions': gradient-based attribution scores for each gene token.
    """
    gradient_data = {}
    model.eval()
    for batch in dataloader:
        emb, scores, cnas, cancer_type, description, times, events, mask, case_ids, gene_names_list = batch
        emb = emb.to(device)
        scores = scores.to(device)
        cnas = cnas.to(device)
        cancer_type = cancer_type.to(device)
        description = description.to(device)
        mask = mask.to(device)
        emb.requires_grad_(True)
        risk = model(emb, scores, cnas, cancer_type, description, src_key_padding_mask=mask)
        risk_sum = risk.sum()
        model.zero_grad()
        risk_sum.backward()
        grads = emb.grad  # [B, seq_len, d_gene]
        attribution = (emb * grads).sum(dim=-1)  # [B, seq_len]
        attribution = attribution.detach().cpu().numpy()
        batch_size = emb.shape[0]
        for i in range(batch_size):
            sample_case = case_ids[i]
            ct_list = cancer_type[i].detach().cpu().tolist()
            sample_ct = None
            for key, binary_vec in cancer_type_mapping.items():
                if binary_vec == [int(x) for x in ct_list]:
                    sample_ct = type_to_index[key]
                    break
            # Skip description token (index 0); gene tokens are indices 1:.
            grad_attr = attribution[i][1:]
            gradient_data[sample_case] = {
                "cancer_type": sample_ct,
                "gene_names": gene_names_list[i],
                "gradient_attributions": grad_attr
            }
    return gradient_data


In [13]:
# %%
# Cell 7: Aggregation Example
def aggregate_attributions(attribution_dict, key="attention_scores"):
    """
    Aggregates attributions by gene across samples.
    Returns a dictionary: gene -> list of (cancer_type, score) tuples.
    """
    aggregated = defaultdict(list)
    for sample in attribution_dict.values():
        for gene, score in zip(sample["gene_names"], sample[key]):
            aggregated[gene].append((sample["cancer_type"], score))
    return aggregated


In [8]:
# %%
# Cell 8: Create Dataset and DataLoader.
# Also load cancer type mapping as in your original code.
CANCER_TYPE_MAPPING_CSV = "/home/chb3333/yulab/chb3333/gem-patho/data_extraction/cancertype_location_description/tcga_study_abbreviations.csv"
df_ct = pd.read_csv(CANCER_TYPE_MAPPING_CSV)
unique_types = sorted(df_ct["Study Abbreviation"].unique())
def int_to_binary_vector(x, width=6):
    return [int(b) for b in format(x, f"0{width}b")]
cancer_type_mapping = {ct: int_to_binary_vector(i, 6) for i, ct in enumerate(unique_types)}
type_to_index = {ct: i for i, ct in enumerate(unique_types)}
print("Cancer type mapping:", cancer_type_mapping)
print("Cancer type to index mapping:", type_to_index)

dataset = PreprocessedSequenceDatasetWithNames(train_df, token_col="gene_embed_seq", cancer_type_mapping=cancer_type_mapping)
dataloader = DataLoader(dataset, batch_size=32, collate_fn=collate_fn_preprocessed_with_names, shuffle=False)


Cancer type mapping: {'ACC': [0, 0, 0, 0, 0, 0], 'BLCA': [0, 0, 0, 0, 0, 1], 'BRCA': [0, 0, 0, 0, 1, 0], 'CESC': [0, 0, 0, 0, 1, 1], 'CHOL': [0, 0, 0, 1, 0, 0], 'CNTL': [0, 0, 0, 1, 0, 1], 'COAD': [0, 0, 0, 1, 1, 0], 'DLBC': [0, 0, 0, 1, 1, 1], 'ESCA': [0, 0, 1, 0, 0, 0], 'FPPP': [0, 0, 1, 0, 0, 1], 'GBM': [0, 0, 1, 0, 1, 0], 'HNSC': [0, 0, 1, 0, 1, 1], 'KICH': [0, 0, 1, 1, 0, 0], 'KIRC': [0, 0, 1, 1, 0, 1], 'KIRP': [0, 0, 1, 1, 1, 0], 'LAML': [0, 0, 1, 1, 1, 1], 'LCML': [0, 1, 0, 0, 0, 0], 'LGG': [0, 1, 0, 0, 0, 1], 'LIHC': [0, 1, 0, 0, 1, 0], 'LUAD': [0, 1, 0, 0, 1, 1], 'LUSC': [0, 1, 0, 1, 0, 0], 'MESO': [0, 1, 0, 1, 0, 1], 'MISC': [0, 1, 0, 1, 1, 0], 'OV': [0, 1, 0, 1, 1, 1], 'PAAD': [0, 1, 1, 0, 0, 0], 'PCPG': [0, 1, 1, 0, 0, 1], 'PRAD': [0, 1, 1, 0, 1, 0], 'READ': [0, 1, 1, 0, 1, 1], 'SARC': [0, 1, 1, 1, 0, 0], 'SKCM': [0, 1, 1, 1, 0, 1], 'STAD': [0, 1, 1, 1, 1, 0], 'TGCT': [0, 1, 1, 1, 1, 1], 'THCA': [1, 0, 0, 0, 0, 0], 'THYM': [1, 0, 0, 0, 0, 1], 'UCEC': [1, 0, 0, 0, 1, 0], 'UC

In [14]:
# %%
# Cell 9: Load the pretrained model and compute attributions.
# First, infer d_gene from one batch.
sample_batch = next(iter(dataloader))
sample_emb = sample_batch[0]
d_gene = sample_emb.shape[-1]

# Instantiate the model.
model = PreprocessedTransformerSurvivalModel(d_gene=d_gene, d_model=256,
                                               polyphen_hidden_dim=128, nhead=4, dropout=0.1,
                                               desc_dim=d_gene)

model_path = "/home/chb3333/yulab/chb3333/gem-patho/models/learning_phases_scheduling/v5_biggerstratbatch/regularizationloss_exp_1_tp_0.8_fp_0.75_warmup_10_transition_10/fold_1/best_model_fold_1.pth"
model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))
print("Pretrained model loaded.")

# ----
# Monkey-patch self-attention layers to force need_weights=True.
for layer in model.transformer_encoder.layers:
    orig_forward = layer.self_attn.forward
    def new_forward(query, key, value, orig_forward=orig_forward, **kwargs):
        kwargs.pop("need_weights", None)  # remove if already provided
        return orig_forward(query, key, value, need_weights=True, **kwargs)
    layer.self_attn.forward = new_forward


# Move model to device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Compute attention-based attributions.
attention_results = get_attention_attributions(model, dataloader, device, cancer_type_mapping, type_to_index)
# Compute gradient-based attributions.
gradient_results  = get_gradient_attributions(model, dataloader, device, cancer_type_mapping, type_to_index)

# Aggregate results by gene.
aggregated_attention = aggregate_attributions(attention_results, key="attention_scores")
aggregated_gradient = aggregate_attributions(gradient_results, key="gradient_attributions")

# Print average attributions per gene.
print("Gradient-Based Attributions (per gene):")
for gene, values in aggregated_gradient.items():
    scores = [score for (_, score) in values]
    print(f"Gene {gene}: Mean gradient attribution = {np.mean(scores):.4f}")

print("\nAttention-Based Attributions (per gene):")
for gene, values in aggregated_attention.items():
    scores = [score for (_, score) in values]
    print(f"Gene {gene}: Mean attention score = {np.mean(scores):.4f}")


/tmp/ipykernel_12411/2619673026.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))


Pretrained model loaded.
Gradient-Based Attributions (per gene):
Gene ARID2: Mean gradient attribution = 0.0011
Gene IDH2: Mean gradient attribution = 0.0235
Gene NBN: Mean gradient attribution = 0.0030
Gene RANBP2: Mean gradient attribution = 0.0003
Gene SETDB1: Mean gradient attribution = 0.0100
Gene TBL1XR1: Mean gradient attribution = 0.0009
Gene TET2: Mean gradient attribution = 0.0031
Gene PRSS8: Mean gradient attribution = -0.0006
Gene ARID1A: Mean gradient attribution = -0.0061
Gene ATIC: Mean gradient attribution = -0.0057
Gene CSMD3: Mean gradient attribution = 0.0016
Gene DICER1: Mean gradient attribution = 0.0055
Gene KRAS: Mean gradient attribution = 0.0271
Gene NFE2L2: Mean gradient attribution = 0.0106
Gene NUMA1: Mean gradient attribution = 0.0008
Gene PIK3CA: Mean gradient attribution = 0.0032
Gene PIK3R1: Mean gradient attribution = 0.0207
Gene PTEN: Mean gradient attribution = 0.0004
Gene ZBTB16: Mean gradient attribution = -0.0030
Gene ATF7IP: Mean gradient attribut

In [16]:
# Compute mean attribution per gene
gene_means = {
    gene: np.mean([score for (_, score) in values])
    for gene, values in aggregated_gradient.items()
}

# Sort by mean attribution
sorted_genes = sorted(gene_means.items(), key=lambda x: x[1], reverse=True)

# Display top 10 (most risk-increasing)
print("\n🔺 Top 20 genes increasing risk:")
for gene, score in sorted_genes[:20]:
    print(f"{gene}: {score:.4f}")

# Display bottom 10 (most risk-decreasing)
print("\n🔻 Top 20 genes decreasing risk:")
for gene, score in sorted_genes[-20:]:
    print(f"{gene}: {score:.4f}")



🔺 Top 20 genes increasing risk:
MTCP1: 0.0330
IDH1: 0.0318
ID3: 0.0274
KRAS: 0.0271
IDH2: 0.0235
LYL1: 0.0217
KIT: 0.0215
SETD1B: 0.0213
NF2: 0.0209
PIK3R1: 0.0207
CDKN2C: 0.0199
PPP2R1A: 0.0197
SMAD4: 0.0195
KEAP1: 0.0189
HOXC13: 0.0189
MDS2: 0.0185
MUC1: 0.0183
EGFR: 0.0181
IL21R: 0.0181
FBXW7: 0.0181

🔻 Top 20 genes decreasing risk:
CALR: -0.0110
CRTC3: -0.0112
SOX17: -0.0112
FAM174B: -0.0116
RMI2: -0.0117
EML4: -0.0117
STK40: -0.0122
FUS: -0.0123
BAX: -0.0135
PBRM1: -0.0156
FEV: -0.0158
TSC2: -0.0163
CD79A: -0.0193
DDIT3: -0.0203
VHL: -0.0223
GNAQ: -0.0223
FUBP1: -0.0233
GNA11: -0.0284
ATRX: -0.0311
CIC: -0.0379


In [17]:
from collections import defaultdict
import numpy as np

# Step 1: Group gene scores by (cancer_type, gene)
per_cancer_gene_scores = defaultdict(list)

for gene, values in aggregated_gradient.items():
    for cancer_type, score in values:
        if cancer_type is not None:
            per_cancer_gene_scores[(cancer_type, gene)].append(score)

# Step 2: Compute mean attribution per (cancer_type, gene)
per_cancer_gene_means = {
    (cancer_type, gene): np.mean(scores)
    for (cancer_type, gene), scores in per_cancer_gene_scores.items()
}


In [20]:
from collections import defaultdict

# Step 3: Build per-cancer dictionaries
genes_by_cancer = defaultdict(list)

for (cancer_type, gene), mean_score in per_cancer_gene_means.items():
    genes_by_cancer[cancer_type].append((gene, mean_score))

# Step 4: Sort & print
for cancer_type, gene_scores in genes_by_cancer.items():
    sorted_genes = sorted(gene_scores, key=lambda x: x[1], reverse=True)
    
    print(f"\n=== Cancer Type {cancer_type} ===")
    print("🔺 Top 10 genes increasing risk:")
    for gene, score in sorted_genes[:10]:
        print(f"  {gene}: {score:.4f}")
    
    print("🔻 Top 10 genes decreasing risk:")
    for gene, score in sorted_genes[-10:]:
        print(f"  {gene}: {score:.4f}")



=== Cancer Type 14 ===
🔺 Top 10 genes increasing risk:
  PIK3CD: 0.2821
  FAM135B: 0.2714
  ZNF429: 0.2527
  SSX1: 0.2463
  NIN: 0.2081
  ERCC3: 0.1699
  SDC4: 0.1676
  SETBP1: 0.1633
  ERG: 0.1627
  NRG1: 0.1496
🔻 Top 10 genes decreasing risk:
  RBM10: -0.1532
  CNOT9: -0.1543
  PEG3: -0.1580
  FAT2: -0.1627
  PDGFRA: -0.1797
  CDKN2A: -0.1879
  ZMYM3: -0.1978
  LSM14A: -0.2007
  RNF6: -0.2090
  HERPUD1: -0.2219

=== Cancer Type 1 ===
🔺 Top 10 genes increasing risk:
  IL10: 0.0896
  FANCL: 0.0657
  MSN: 0.0505
  GNA12: 0.0458
  CD58: 0.0427
  RNF43: 0.0414
  NCAPH: 0.0402
  MSH3: 0.0360
  FOXF1: 0.0333
  JARID2: 0.0324
🔻 Top 10 genes decreasing risk:
  SOX9: -0.0246
  HEY1: -0.0255
  PWWP2A: -0.0274
  SMARCB1: -0.0282
  CD19: -0.0287
  GNAQ: -0.0425
  GPC3: -0.0523
  AIP: -0.0526
  PTK6: -0.0601
  ZNF395: -0.0711

=== Cancer Type 29 ===
🔺 Top 10 genes increasing risk:
  PTPN1: 0.0713
  MDM4: 0.0638
  ARFRP1: 0.0568
  GNAQ: 0.0544
  PRKAR1A: 0.0341
  KCNJ5: 0.0251
  MAP2K4: 0.0243
  K